# 01 · Pengumpulan & Cleaning Data

Notebook ini bertugas mengubah 6 file Excel mentah dari BPS (`data/raw/`) menjadi **1 tabel master bersih** (`data/processed/indikator_digital_provinsi.csv`) dengan kerangka penuh **38 provinsi**.

Alurnya:
1. Cek file mentah & intip format header khas BPS.
2. Bersihkan 4 tabel standar (internet, HP, komputer, keterampilan TIK).
3. Hitung persentase desa 4G/5G dari data Podes (jumlah desa → persen).
4. Filter baris provinsi pada data RLS (memisahkan dari kab/kota).
5. Gabungkan (merge) keenam tabel, validasi kualitas data, lalu simpan ke CSV.

In [10]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.config import DATA_PROCESSED, DATA_RAW, FEATURES, MASTER_FILE, PROVINSI_38
from src.preprocessing import load_bps_table, merge_indicators, normalize_provinsi

## 1. Cek File Mentah di `data/raw/`

In [11]:
raw_files = sorted(p.name for p in DATA_RAW.iterdir() if p.name != ".gitkeep")
print(f"Jumlah file mentah: {len(raw_files)}")
raw_files

Jumlah file mentah: 6


['pct_desa_4G.xlsx',
 'pct_hp.xlsx',
 'pct_internet.xlsx',
 'pct_komputer.xlsx',
 'pct_skill_tik.xlsx',
 'rls.xlsx']

## 2. Intip Struktur File Mentah BPS
File unduhan BPS umumnya memiliki beberapa baris judul dan sub-header bertingkat sebelum data `ACEH` dimulai. Kita intip satu contohnya agar tahu berapa baris yang perlu dilewati (`skiprows`).

In [12]:
pd.read_excel(DATA_RAW / "pct_internet.xlsx").head(7)

,38 Provinsi,Unnamed: 1,Unnamed: 2,Unnamed: 3
0,NaN,Persentase Penduduk Berumur 5 Tahun ke Atas ya...,NaN,NaN
1,NaN,Perkotaan,Perdesaan,Perkotaan+Perdesaan
2,NaN,2025,2025,2025
3,ACEH,77.51,65.87,70.03
4,SUMATERA UTARA,82.53,75.52,79.6
5,SUMATERA BARAT,84.93,75.52,80.47
6,RIAU,87.5,76.58,80.99


## 3. Bersihkan 4 Tabel Standar (Internet, HP, Komputer, Keterampilan TIK)
- `pct_internet.xlsx`, `pct_hp.xlsx`, `pct_komputer.xlsx` (2025): lewati 4 baris header (`skiprows=4, header=None`), ambil kolom `0` (provinsi) dan kolom `3` (`Perkotaan+Perdesaan`).
- `pct_skill_tik.xlsx` (2025): lewati 3 baris header (`skiprows=3, header=None`), ambil kolom `0` (provinsi) dan kolom `1` (persentase).

In [13]:
df_internet = load_bps_table(
    DATA_RAW / "pct_internet.xlsx",
    prov_col=0, value_col=3, new_name="pct_internet",
    skiprows=4, header=None,
)
df_hp = load_bps_table(
    DATA_RAW / "pct_hp.xlsx",
    prov_col=0, value_col=3, new_name="pct_hp",
    skiprows=4, header=None,
)
df_komputer = load_bps_table(
    DATA_RAW / "pct_komputer.xlsx",
    prov_col=0, value_col=3, new_name="pct_komputer",
    skiprows=4, header=None,
)
df_skill = load_bps_table(
    DATA_RAW / "pct_skill_tik.xlsx",
    prov_col=0, value_col=1, new_name="pct_skill_tik",
    skiprows=3, header=None,
)

for label, d in [("internet", df_internet), ("hp", df_hp), ("komputer", df_komputer), ("skill_tik", df_skill)]:
    print(f"{label:10s} -> shape={d.shape}, missing={int(d.isna().sum().sum())}")

internet   -> shape=(38, 2), missing=0
hp         -> shape=(38, 2), missing=0
komputer   -> shape=(38, 2), missing=0
skill_tik  -> shape=(38, 2), missing=0


## 4. Bersihkan & Hitung Persentase Desa Sinyal 4G/5G (`pct_desa_4G.xlsx`)
Data Podes menyimpan **jumlah desa** untuk 4 kategori sinyal dan memakai simbol `–` untuk nol. Kita ubah simbol `–` menjadi `0`, jumlahkan total desa per provinsi, lalu hitung persentase desa yang sudah terjangkau 4G/5G.

In [14]:
raw_4g = pd.read_excel(DATA_RAW / "pct_desa_4G.xlsx")
raw_4g.columns = ["provinsi_raw", "desa_3g", "desa_25g", "desa_tidak_ada", "desa_4g_5g"]

# Normalisasi nama provinsi dan buang baris non-provinsi ('Indonesia', catatan kaki)
raw_4g["provinsi"] = raw_4g["provinsi_raw"].map(normalize_provinsi)
raw_4g = raw_4g.dropna(subset=["provinsi"]).copy()

# Ubah kolom jumlah desa jadi angka (simbol '–' menjadi NaN lalu diisi 0)
signal_cols = ["desa_3g", "desa_25g", "desa_tidak_ada", "desa_4g_5g"]
for col in signal_cols:
    raw_4g[col] = pd.to_numeric(raw_4g[col], errors="coerce").fillna(0)

# Hitung persentase desa dengan sinyal 5G/4G/LTE
raw_4g["total_desa"] = raw_4g[signal_cols].sum(axis=1)
raw_4g["pct_desa_4g"] = (raw_4g["desa_4g_5g"] / raw_4g["total_desa"] * 100).round(2)

df_4g = raw_4g[["provinsi", "pct_desa_4g"]].reset_index(drop=True)
print(f"desa_4g    -> shape={df_4g.shape}, missing={int(df_4g.isna().sum().sum())}")
df_4g.head()

desa_4g    -> shape=(38, 2), missing=0


,provinsi,pct_desa_4g
0,Aceh,89.67
1,Sumatera Utara,87.68
2,Sumatera Barat,93.55
3,Riau,91.44
4,Jambi,90.23


## 5. Bersihkan Rata-rata Lama Sekolah (`rls.xlsx`)
File `rls.xlsx` (2026) memuat 582 baris gabungan provinsi dan kabupaten/kota. Karena hanya baris **provinsi** (dan `INDONESIA`) yang ditulis HURUF KAPITAL, kita filter `.str.isupper()` lebih dulu sebelum normalisasi agar kabupaten/kota yang namanya sama dengan provinsi (seperti Gorontalo dan Bengkulu) tidak ikut masuk.

In [15]:
raw_rls = pd.read_excel(DATA_RAW / "rls.xlsx", skiprows=3, header=None)

# Ambil hanya baris provinsi (ditulis HURUF KAPITAL di tabel BPS)
raw_rls = raw_rls[raw_rls[0].astype(str).str.isupper()].copy()

df_rls = pd.DataFrame({
    "provinsi": raw_rls[0].map(normalize_provinsi),
    "rls": pd.to_numeric(raw_rls[1].astype(str).str.replace(",", ".", regex=False), errors="coerce"),
}).dropna(subset=["provinsi"]).reset_index(drop=True)

print(f"rls        -> shape={df_rls.shape}, missing={int(df_rls.isna().sum().sum())}")
df_rls.head()

rls        -> shape=(38, 2), missing=0


,provinsi,rls
0,Aceh,10.26
1,Sumatera Utara,10.13
2,Sumatera Barat,10.03
3,Riau,9.56
4,Jambi,9.18


## 6. Gabungkan Semua Indikator (Merge) & Validasi Kualitas Data
Keenam tabel digabung ke kerangka baku `PROVINSI_38` menggunakan kunci kolom `provinsi`, lalu dicek dengan `assert` agar dipastikan jumlah baris tepat 38, tidak ada duplikat, dan tidak ada *missing value*.

In [16]:
tables = [df_internet, df_hp, df_komputer, df_4g, df_skill, df_rls]
master = merge_indicators(tables, PROVINSI_38)

# Sanity check otomatis
assert len(master) == 38, f"Harusnya 38 provinsi, dapat {len(master)}"
assert master["provinsi"].nunique() == 38, "Ada provinsi duplikat!"
assert master[FEATURES].isna().sum().sum() == 0, "Masih ada missing value!"

print("Validasi lolos! Ukuran tabel master:", master.shape)
print("\nJumlah missing value per kolom:")
print(master.isna().sum())
master.describe().T[["min", "mean", "50%", "max"]].round(2)

Validasi lolos! Ukuran tabel master: (38, 7)

Jumlah missing value per kolom:
provinsi         0
pct_internet     0
pct_hp           0
pct_komputer     0
pct_desa_4g      0
pct_skill_tik    0
rls              0
dtype: int64


,min,mean,50%,max
pct_internet,15.40,73.21,76.57,91.32
pct_hp,22.49,68.99,70.18,84.47
pct_komputer,1.96,19.93,18.46,38.42
pct_desa_4g,19.53,85.45,90.08,100.00
pct_skill_tik,15.79,90.68,97.22,99.54
rls,4.31,9.19,9.22,11.76


In [17]:
# Intip 5 provinsi pertama & 5 provinsi terakhir (termasuk pemekaran Papua)
pd.concat([master.head(5), master.tail(5)])

,provinsi,pct_internet,pct_hp,pct_komputer,pct_desa_4g,pct_skill_tik,rls
0,Aceh,70.03,68.21,17.59,89.67,93.72,10.26
1,Sumatera Utara,79.60,70.40,16.32,87.68,96.78,10.13
2,Sumatera Barat,80.47,70.33,22.21,93.55,97.15,10.03
3,Riau,80.99,76.17,18.54,91.44,97.29,9.56
4,Jambi,76.27,71.03,18.39,90.23,97.39,9.18
33,Papua Barat Daya,70.36,69.08,23.97,88.64,82.98,9.03
34,Papua,64.64,68.56,23.14,74.40,78.79,10.27
35,Papua Selatan,50.02,50.67,17.34,86.38,57.49,8.65
36,Papua Tengah,32.64,37.35,7.02,20.94,37.06,6.14
37,Papua Pegunungan,15.40,22.49,1.96,19.53,15.79,4.31


## 7. Simpan Data Master ke `data/processed/`

In [18]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
master.to_csv(MASTER_FILE, index=False)
print(f"Berhasil disimpan ke: {MASTER_FILE}")
print(f"Ukuran akhir: {master.shape[0]} provinsi x {master.shape[1]} kolom")

Berhasil disimpan ke: C:\Users\USER\Documents\indonesia_digital_divide_clustering\data\processed\indikator_digital_provinsi.csv
Ukuran akhir: 38 provinsi x 7 kolom
